# Augmented Training — Whisper LoRA + Small (Kaggle)

Re-trains both Whisper-large-v3 LoRA and Whisper-small on noise-augmented data.
Uses RealClass classroom noise + MUSAN babble noise from S4.1 augmentation pipeline.

**Requirements:**
- Kaggle dataset with competition audio uploaded as a private dataset
- RealClass noise dataset (from DrivenData)
- MUSAN noise subset (babble/noise)
- HuggingFace Hub token (set as Kaggle secret `HF_TOKEN`)
- ~16 hours GPU quota (8 hrs LoRA + 6 hrs small + overhead)

## 1. Setup

In [ ]:
import subprocess
import sys

# Install dependencies not pre-installed on Kaggle
deps = ["jiwer>=3.0", "audiomentations>=0.36", "peft>=0.12", "bitsandbytes>=0.43"]
for dep in deps:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", dep])

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path(".").resolve()
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif Path("/kaggle/input/childwhisper-src/src").exists():
    sys.path.insert(0, "/kaggle/input/childwhisper-src")
    PROJECT_ROOT = Path("/kaggle/input/childwhisper-src")

from src.kaggle_utils import (
    get_paths_lora,
    setup_hub_auth,
    verify_kaggle_data,
    check_gpu_memory,
    is_kaggle,
)
from src.train_whisper_lora import main as train_lora_main
from src.train_whisper_small import main as train_small_main

print(f"Running on Kaggle: {is_kaggle()}")
print(f"Project root: {PROJECT_ROOT}")

## 2. Environment & Paths

In [ ]:
# Configure paths based on environment
DATASET_SLUG = "pasketti-word-audio"
LOCAL_DATA_DIR = "data"

paths = get_paths_lora(dataset_slug=DATASET_SLUG, local_data_dir=LOCAL_DATA_DIR)
AUDIO_DIR = paths["audio_dir"]
METADATA_PATH = paths["metadata_path"]

# Noise directories
if is_kaggle():
    NOISE_DIR = Path("/kaggle/input/musan-noise/noise")
    REALCLASS_DIR = Path("/kaggle/input/realclass-noise/realclass")
else:
    NOISE_DIR = Path("data/musan_noise")
    REALCLASS_DIR = Path("data/realclass_noise")

# Output directories (separate from clean models)
LORA_OUTPUT_DIR = Path(paths["output_dir"]).parent / "whisper-large-v3-lora-augmented"
SMALL_OUTPUT_DIR = Path(paths["output_dir"]).parent / "whisper-small-augmented"

CONFIG_PATH = PROJECT_ROOT / "configs" / "training_config.yaml"
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("configs/training_config.yaml")

print(f"Audio dir:       {AUDIO_DIR}")
print(f"Metadata:        {METADATA_PATH}")
print(f"Noise dir:       {NOISE_DIR}")
print(f"RealClass dir:   {REALCLASS_DIR}")
print(f"LoRA output:     {LORA_OUTPUT_DIR}")
print(f"Small output:    {SMALL_OUTPUT_DIR}")
print(f"Config:          {CONFIG_PATH}")

## 3. Data & Noise Verification

In [ ]:
# Verify competition data
stats = verify_kaggle_data(str(AUDIO_DIR), str(METADATA_PATH))
print(f"Utterances:       {stats['num_utterances']}")
print(f"Audio found:      {stats['num_audio_found']}")
print(f"Audio missing:    {stats['num_missing_audio']}")

# Verify noise directories
noise_files = list(NOISE_DIR.glob("*.wav")) if NOISE_DIR.exists() else []
realclass_files = list(REALCLASS_DIR.glob("*.wav")) if REALCLASS_DIR.exists() else []
print(f"\nNoise files:      {len(noise_files)} in {NOISE_DIR}")
print(f"RealClass files:  {len(realclass_files)} in {REALCLASS_DIR}")

assert NOISE_DIR.exists(), f"Noise dir not found: {NOISE_DIR}"
assert REALCLASS_DIR.exists(), f"RealClass dir not found: {REALCLASS_DIR}"
assert len(noise_files) > 0, "No noise files found"
assert len(realclass_files) > 0, "No RealClass files found"

## 4. HuggingFace Hub Authentication

In [ ]:
LORA_HUB_MODEL_ID = "nishantgaurav23/pasketti-whisper-lora-augmented"
SMALL_HUB_MODEL_ID = "nishantgaurav23/pasketti-whisper-small-augmented"
PUSH_TO_HUB = True

try:
    setup_hub_auth()
    print("HF Hub authenticated successfully")
except ValueError as e:
    print(f"Hub auth skipped: {e}")
    print("Checkpoints will be saved locally only")
    PUSH_TO_HUB = False

## 5. GPU Check

In [ ]:
gpu_info = check_gpu_memory()
print(f"GPU:              {gpu_info['gpu_name'] or 'None (CPU only)'}")
print(f"Memory:           {gpu_info['total_memory_gb']:.1f} GB")
print(f"Sufficient:       {gpu_info['is_sufficient']}")

## 6. Train Whisper-large-v3 LoRA (Augmented)

In [ ]:
import time

lora_args = [
    "--metadata-path", str(METADATA_PATH),
    "--audio-dir", str(AUDIO_DIR),
    "--config", str(CONFIG_PATH),
    "--output-dir", str(LORA_OUTPUT_DIR),
    "--noise-dir", str(NOISE_DIR),
    "--realclass-dir", str(REALCLASS_DIR),
    "--hub-model-id", LORA_HUB_MODEL_ID,
]
if not PUSH_TO_HUB:
    lora_args.append("--no-push-to-hub")

print(f"LoRA training args: {' '.join(lora_args)}")
print("\nStarting augmented LoRA training...")

start_time = time.time()
lora_wer = train_lora_main(lora_args)
lora_elapsed = time.time() - start_time

print(f"\nLoRA training complete!")
print(f"Validation WER: {lora_wer:.4f}")
print(f"Elapsed time: {lora_elapsed / 60:.1f} minutes")

## 7. Train Whisper-small (Augmented)

In [ ]:
small_args = [
    "--metadata-path", str(METADATA_PATH),
    "--audio-dir", str(AUDIO_DIR),
    "--config", str(CONFIG_PATH),
    "--output-dir", str(SMALL_OUTPUT_DIR),
    "--noise-dir", str(NOISE_DIR),
    "--realclass-dir", str(REALCLASS_DIR),
    "--hub-model-id", SMALL_HUB_MODEL_ID,
]
if not PUSH_TO_HUB:
    small_args.append("--no-push-to-hub")

print(f"Small training args: {' '.join(small_args)}")
print("\nStarting augmented Whisper-small training...")

start_time = time.time()
small_wer = train_small_main(small_args)
small_elapsed = time.time() - start_time

print(f"\nWhisper-small training complete!")
print(f"Validation WER: {small_wer:.4f}")
print(f"Elapsed time: {small_elapsed / 60:.1f} minutes")

## 8. Summary

In [ ]:
total_elapsed = lora_elapsed + small_elapsed

print("=== Augmented Training Summary ===")
print(f"LoRA WER:         {lora_wer:.4f} ({lora_elapsed / 60:.1f} min)")
print(f"Small WER:        {small_wer:.4f} ({small_elapsed / 60:.1f} min)")
print(f"Total time:       {total_elapsed / 60:.1f} min ({total_elapsed / 3600:.1f} hrs)")
print(f"\nAugmented LoRA hub:  {LORA_HUB_MODEL_ID}")
print(f"Augmented Small hub: {SMALL_HUB_MODEL_ID}")

best_wer = min(lora_wer, small_wer)
if best_wer < 0.17:
    print("\nExcellent! Target WER (<0.17) achieved with augmented training!")
elif best_wer < 0.20:
    print("\nGood progress. Consider Phase 5 optimizations for further improvement.")
else:
    print("\nWER still high. Check noise data quality or augmentation SNR ranges.")